In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import split, explode, trim
from pyspark.sql.functions import regexp_replace, col
from pyspark.sql.functions import year, month, dayofmonth, to_timestamp
import os
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"

In [2]:
spark = SparkSession.builder \
    .appName("ReadCSV") \
    .getOrCreate()

In [3]:
raw_data_set = spark.read.csv(
    r"C:\Users\patsi\Documents\Virtual_data_department\Data_set\Retail_Transactions_Dataset.csv",
    header=True,
    inferSchema=True
)

In [4]:
# 3. Split products into array
df_split = df.withColumn(
    "Product",
    split("Product", ",")
)

# 4. Explode into rows
df_exploded = df_split.withColumn(
    "Product",
    explode("Product")
)

# 5. Clean whitespace (best practice)
df_final = df_exploded.withColumn("Product", trim("Product")) \
                      .select("Transaction_ID", "Product")

# 6. Show result
#df_final.show()

In [5]:
def keep_alphanumerics_and_spaces(df, column_name, new_column_name=None):
    if new_column_name is None:
        new_column_name = column_name
    return df.withColumn(
        new_column_name,
        regexp_replace(col(column_name), r"[^a-zA-Z0-9 ]", "")
    )

In [6]:
df_clean = keep_alphanumerics_and_spaces(df_final, "Product", "Product_Clean")

In [7]:
#pandas_df = df_clean.toPandas()

In [8]:
df_clean.select("Product_Clean").dropDuplicates().show()

+--------------+
| Product_Clean|
+--------------+
|       Pickles|
|        Shrimp|
|       Ketchup|
|         Water|
|     Olive Oil|
|         Broom|
|      Potatoes|
|        Butter|
|        Cereal|
|       Mustard|
|           Tea|
|    Lawn Mower|
|Hand Sanitizer|
|          Soap|
|        Razors|
|        Yogurt|
|        Orange|
|        Banana|
|          Milk|
|        Onions|
+--------------+
only showing top 20 rows


In [26]:
#pandas_df['Product_Clean'].drop_duplicates().to_csv(r"C:\Users\patsi\Documents\Virtual_data_department\Data_engineering\Extract\products_pricing_matrix.csv",index=False)

In [28]:
#pandas_df['Product_Clean'].sort_values()

1807096    Air Freshener
2653822    Air Freshener
129213     Air Freshener
2066198    Air Freshener
789046     Air Freshener
               ...      
2393980           Yogurt
174037            Yogurt
23292             Yogurt
339223            Yogurt
2564285           Yogurt
Name: Product_Clean, Length: 3000343, dtype: object

In [32]:
df.show(1)

+--------------+-------------------+-------------+--------------------+-----------+----------+--------------+-----------+--------------+----------------+-----------------+------+---------+----+-----+---+
|Transaction_ID|               Date|Customer_Name|             Product|Total_Items|Total_Cost|Payment_Method|       City|    Store_Type|Discount_Applied|Customer_Category|Season|Promotion|year|month|day|
+--------------+-------------------+-------------+--------------------+-----------+----------+--------------+-----------+--------------+----------------+-----------------+------+---------+----+-----+---+
|    1000000000|2022-01-21 06:27:29| Stacey Price|['Ketchup', 'Shav...|          3|     71.65|Mobile Payment|Los Angeles|Warehouse Club|            true|        Homemaker|Winter|     None|2022|    1| 21|
+--------------+-------------------+-------------+--------------------+-----------+----------+--------------+-----------+--------------+----------------+-----------------+------+------

In [33]:
df = df.withColumn(
    "Date",
    to_timestamp("Date")
)

df = df.withColumn("year", year("Date")) \
       .withColumn("month", month("Date")) \
       .withColumn("day", dayofmonth("Date"))

In [10]:
pricing_matrix = spark.read.csv(
    r"C:\Users\patsi\Documents\Virtual_data_department\Data_engineering\Extract\products_pricing_matrix.csv",
    header=True,
    inferSchema=True
)

In [11]:
pricing_matrix.show()

+-------------+-----+
|Product_Clean|Price|
+-------------+-----+
|      Ketchup|   30|
|Shaving Cream|   40|
|  Light Bulbs|   15|
|    Ice Cream|   30|
|         Milk|   22|
|    Olive Oil|   80|
|        Bread|   16|
|     Potatoes|   65|
|      Spinach|   11|
|      Tissues|    7|
|      Mustard|   34|
|    Dish Soap|   25|
|   Toothpaste|   12|
|      Chicken|   45|
|        Honey|   30|
|    BBQ Sauce|   60|
|         Soda|   20|
|  Garden Hose|  300|
|        Syrup|   35|
|   Trash Cans|  405|
+-------------+-----+
only showing top 20 rows
